### Import packages and define paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
from gensim.models.word2vec import Word2Vec
from gensim.matutils import unitvec

In [2]:
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
EMBEDDINGS_DIR = PROJECT_ROOT / "embeddings"
DATA_DIR = PROJECT_ROOT / "data/dictionaries"
GENDERED_NAMES_DIR = PROJECT_ROOT / "data/gendered_names"
TARGET_WORDS_DIR = PROJECT_ROOT / "data/target_words"
OUTPUT_DIR = PROJECT_ROOT / "data/associational_gender_bias"

warmth_competence_lexicon_path = DATA_DIR / "Seed Dictionaries.csv"

lexicon = pd.read_csv(warmth_competence_lexicon_path)

In [3]:
warmth_competence_lexicon_path = DATA_DIR / "Seed Dictionaries.csv"

lexicon = pd.read_csv(warmth_competence_lexicon_path)

### Load embeddings

In [4]:
seeds = [1, 7, 13, 42, 99, 123, 256, 512, 777, 999]

romance_models = {
    seed: Word2Vec.load(str(EMBEDDINGS_DIR / f"romance_{seed}.w2v"))
    for seed in seeds
}

sci_fi_models = {
    seed: Word2Vec.load(str(EMBEDDINGS_DIR / f"sci_fi_{seed}.w2v"))
    for seed in seeds
}

lit_models = {
    seed: Word2Vec.load(str(EMBEDDINGS_DIR / f"literary_fiction_{seed}.w2v"))
    for seed in seeds
}

### Create average gender vectors

In [5]:
# Load previously extracted gendered names
with open(GENDERED_NAMES_DIR / "male_names_sci_fi.json") as f:
    male_names_sci_fi = json.load(f)

with open(GENDERED_NAMES_DIR / "female_names_sci_fi.json") as f:
    female_names_sci_fi = json.load(f)

with open(GENDERED_NAMES_DIR / "male_names_lit.json") as f:
    male_names_lit = json.load(f)

with open(GENDERED_NAMES_DIR / "female_names_lit.json") as f:
    female_names_lit = json.load(f)

with open(GENDERED_NAMES_DIR / "male_names_romance.json") as f:
    male_names_romance = json.load(f)
    
with open(GENDERED_NAMES_DIR / "female_names_romance.json") as f:
    female_names_romance = json.load(f)

In [6]:
female_list = ['she', 'daughter', 'hers', 'her', 'mother', 'woman', 'girl', 'herself', 'female', 'sister', 'daughters', 'mothers', 'women', 'girls', 'females', 'sisters', 'aunt', 'aunts', 'niece', 'nieces', 'mrs.', 'miss']
male_list = ['he', 'son', 'his', 'him', 'father', 'man', 'boy', 'himself', 'male', 'brother', 'sons', 'fathers', 'men', 'boys', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews', 'mr.']

In [7]:
# combine gender list with names for each genre
female_list_sci_fi = female_list + female_names_sci_fi
male_list_sci_fi = male_list + male_names_sci_fi

female_list_lit = female_list + female_names_lit
male_list_lit = male_list + male_names_lit

female_list_romance = female_list + female_names_romance 
male_list_romance = male_list + male_names_romance

In [8]:
# Function to compute the average vector
def compute_average_vector(model, word_list):
    vectors = [model.wv[word] for word in word_list if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        raise ValueError(f"None of the words in {word_list} exist in the model.")

In [9]:
# helper function to compute cosine similarity in update_gender_vectors before and after update
def cosine_sim(a, b):
    return np.dot(unitvec(a), unitvec(b))

In [10]:
# function that updates a vector and compares it to the original, then saves the updated model to a new file
def update_gender_vectors(model, male_words, female_words, genre_name, save_dir):
    # Save original vectors
    original_he = model.wv["he"].copy()
    original_she = model.wv["she"].copy()

    # Compute new vectors
    he_vector = compute_average_vector(model, male_words)
    she_vector = compute_average_vector(model, female_words)

    # Safe update
    model.wv["he"] = he_vector
    model.wv["she"] = she_vector

    # Compare with original
    he_sim = cosine_sim(original_he, model.wv["he"])
    she_sim = cosine_sim(original_she, model.wv["she"])
    print(f"{genre_name} — he similarity with original: {he_sim:.6f}")
    print(f"{genre_name} — she similarity with original: {she_sim:.6f}")

    # Save the updated model to a new file
    save_path = save_dir / f"{genre_name}_with_gender_vectors.w2v"
    model.save(str(save_path))

    return model

In [11]:
models = {
    "romance": romance_models,
    "sci_fi": sci_fi_models,
    "literary_fiction": lit_models
}

name_lists = {
    "romance": (male_list_romance, female_list_romance),
    "sci_fi": (male_list_sci_fi, female_list_sci_fi),
    "literary_fiction": (male_list_lit, female_list_lit)
}

seeds = [1, 7, 13, 42, 99, 123, 256, 512, 777, 999]

In [12]:
for genre, genre_models in models.items():
    male_list, female_list = name_lists[genre]

    for seed, model in genre_models.items():
        genre_seed_name = f"{genre}_{seed}"

        update_gender_vectors(
            model,
            male_list,
            female_list,
            genre_seed_name,
            EMBEDDINGS_DIR
        )

romance_1 — he similarity with original: 0.650725
romance_1 — she similarity with original: 0.668046
romance_7 — he similarity with original: 0.656740
romance_7 — she similarity with original: 0.660893
romance_13 — he similarity with original: 0.672136
romance_13 — she similarity with original: 0.663208
romance_42 — he similarity with original: 0.673649
romance_42 — she similarity with original: 0.663774
romance_99 — he similarity with original: 0.663865
romance_99 — she similarity with original: 0.668964
romance_123 — he similarity with original: 0.654166
romance_123 — she similarity with original: 0.646367
romance_256 — he similarity with original: 0.684717
romance_256 — she similarity with original: 0.662190
romance_512 — he similarity with original: 0.670715
romance_512 — she similarity with original: 0.682205
romance_777 — he similarity with original: 0.636446
romance_777 — she similarity with original: 0.644397
romance_999 — he similarity with original: 0.655275
romance_999 — she

In [13]:
updated_models = {
    genre: {
        seed: Word2Vec.load(
            str(EMBEDDINGS_DIR / f"{genre}_{seed}_with_gender_vectors.w2v")
        )
        for seed in seeds
    }
    for genre in models.keys()
}

In [14]:
# Reload original embeddings so we start from a clean baseline
# Embeddings without names 
romance_models_no_names = {
    seed: Word2Vec.load(str(EMBEDDINGS_DIR / f"romance_{seed}.w2v"))
    for seed in seeds
}

sci_fi_models_no_names = {
    seed: Word2Vec.load(str(EMBEDDINGS_DIR / f"sci_fi_{seed}.w2v"))
    for seed in seeds
}

lit_models_no_names = {
    seed: Word2Vec.load(str(EMBEDDINGS_DIR / f"literary_fiction_{seed}.w2v"))
    for seed in seeds
}

models_no_names = {
    "romance": romance_models_no_names,
    "sci_fi": sci_fi_models_no_names,
    "literary_fiction": lit_models_no_names
}

for genre, genre_models in models_no_names.items():

    for seed, model in genre_models.items():
        genre_seed_name = f"{genre}_{seed}_no_names"

        update_gender_vectors(
            model,
            male_list,
            female_list,
            genre_seed_name,
            EMBEDDINGS_DIR
        )



romance_1_no_names — he similarity with original: 0.653318
romance_1_no_names — she similarity with original: 0.670835
romance_7_no_names — he similarity with original: 0.660218
romance_7_no_names — she similarity with original: 0.663300
romance_13_no_names — he similarity with original: 0.675492
romance_13_no_names — she similarity with original: 0.667213
romance_42_no_names — he similarity with original: 0.677545
romance_42_no_names — she similarity with original: 0.668785
romance_99_no_names — he similarity with original: 0.667691
romance_99_no_names — she similarity with original: 0.672567
romance_123_no_names — he similarity with original: 0.658612
romance_123_no_names — she similarity with original: 0.648080
romance_256_no_names — he similarity with original: 0.688958
romance_256_no_names — she similarity with original: 0.666473
romance_512_no_names — he similarity with original: 0.673731
romance_512_no_names — she similarity with original: 0.686811
romance_777_no_names — he simi

In [15]:
updated_models_no_names = {
    genre: {
        seed: Word2Vec.load(
            str(EMBEDDINGS_DIR / f"{genre}_{seed}_no_names_with_gender_vectors.w2v")
        )
        for seed in seeds
    }
    for genre in models.keys()
}

### Define target dimensions/word categories

In [16]:
# wamrth is sociability and morality
high_sociability_words = lexicon[(lexicon["Dictionary"] == "Sociability") & (lexicon["Dir"] == "high")]["term"].tolist()
low_sociability_words = lexicon[(lexicon["Dictionary"] == "Sociability") & (lexicon["Dir"] == "low")]["term"].tolist()
high_morality_words = lexicon[(lexicon["Dictionary"] == "Morality") & (lexicon["Dir"] == "high")]["term"].tolist()
low_morality_words = lexicon[(lexicon["Dictionary"] == "Morality") & (lexicon["Dir"] == "low")]["term"].tolist()

# competence is agency and ability
high_agency_words = lexicon[(lexicon["Dictionary"] == "Agency") & (lexicon["Dir"] == "high")]["term"].tolist()
low_agency_words = lexicon[(lexicon["Dictionary"] == "Agency") & (lexicon["Dir"] == "low")]["term"].tolist()
high_ability_words = lexicon[(lexicon["Dictionary"] == "Ability") & (lexicon["Dir"] == "high")]["term"].tolist()
low_ability_words = lexicon[(lexicon["Dictionary"] == "Ability") & (lexicon["Dir"] == "low")]["term"].tolist()

In [17]:
# Remove duplicates within each combined list, preserving order
high_warmth     = list(dict.fromkeys(high_sociability_words + high_morality_words))
low_warmth      = list(dict.fromkeys(low_sociability_words + low_morality_words))
high_competence = list(dict.fromkeys(high_agency_words + high_ability_words))
low_competence  = list(dict.fromkeys(low_agency_words + low_ability_words))

In [18]:
# check amount of words in each list
len(high_competence), len(low_competence), len(high_warmth), len(low_warmth)

(68, 60, 74, 81)

### Check missing words and make specific category list for each genre depending on missing words

In [19]:
# defining possible genres
genre_embeddings = {
    "romance": romance_models[1],
    "sci_fi": sci_fi_models[1],
    "literary_fiction": lit_models[1]
}

# defining dimensions/word lists
word_lists = {
    "high_competence": high_competence,
    "low_competence": low_competence,
    "high_warmth": high_warmth,
    "low_warmth": low_warmth
}
# Function to create genre-specific filtered lists
# this is to avoid issues with missing words
def create_genre_specific_lists(genre_embeddings, word_lists):
    filtered_lists = {}
    missing_summary = {}
    
    for genre_name, emb in genre_embeddings.items():
        for category_name, words in word_lists.items():
            # Filter words present in this genre's embeddings
            filtered = [w for w in words if w in emb.wv]
            missing = [w for w in words if w not in emb.wv]
            
            key = f"{category_name}_{genre_name}"
            filtered_lists[key] = filtered
            missing_summary[key] = missing
            
    return filtered_lists, missing_summary


In [20]:
# Apply the function
filtered_genre_lists, missing_genre_words = create_genre_specific_lists(genre_embeddings, word_lists)

# access the filtered lists
high_competence_romance = filtered_genre_lists["high_competence_romance"]
high_competence_sci_fi = filtered_genre_lists["high_competence_sci_fi"]
high_competence_literary_fiction = filtered_genre_lists["high_competence_literary_fiction"]

low_competence_romance = filtered_genre_lists["low_competence_romance"]
low_competence_sci_fi = filtered_genre_lists["low_competence_sci_fi"]
low_competence_literary_fiction = filtered_genre_lists["low_competence_literary_fiction"]

high_warmth_romance = filtered_genre_lists["high_warmth_romance"]
high_warmth_sci_fi = filtered_genre_lists["high_warmth_sci_fi"]
high_warmth_literary_fiction = filtered_genre_lists["high_warmth_literary_fiction"]

low_warmth_romance = filtered_genre_lists["low_warmth_romance"]
low_warmth_sci_fi = filtered_genre_lists["low_warmth_sci_fi"]
low_warmth_literary_fiction = filtered_genre_lists["low_warmth_literary_fiction"]

# inspect missing words
for key, missing in missing_genre_words.items():
    if missing:
        print(f"{key}: {len(missing)} missing words -> {missing}")

high_competence_romance: 28 missing words -> ['fearlessness', 'assertiveness', 'assertive', 'unassertive', 'striver', 'striving', 'industrious', 'energetic', 'self-confident', 'self-reliant', 'conscientious', 'motivated', 'unwavering', 'autonomous', 'dominating', 'dominant', 'dominance', 'competitive', 'smart', 'intelligent', 'skillful', 'educated', 'felicitous', 'imaginative', 'shrewd', 'discriminating', 'inventive', 'insightful']
low_competence_romance: 41 missing words -> ['diffident', 'unassertiveness', 'insecure', 'inactive', 'doubtful', 'dependent', 'apathy', 'unenterprising', 'negligent', 'lethargic', 'unambitious', 'undedicated', 'unadventurous', 'unmotivated', 'nonresilient', 'spiritless', 'dominated', 'submissive', 'meek', 'docile', 'incompetent', 'uncompetitive', 'unintelligent', 'stupidity', 'ignorant', 'ignorance', 'dumb', 'dumbness', 'uneducated', 'uncreative', 'incapable', 'unimaginative', 'undiscriminating', 'maladroit', 'folly', 'unwise', 'inefficient', 'ineffective', 

In [21]:
# Combine categories
all_categories_romance = {
    "high_competence": high_competence_romance,
    "low_competence": low_competence_romance,
    "high_warmth": high_warmth_romance,
    "low_warmth": low_warmth_romance
}

# save as json
with open(TARGET_WORDS_DIR / "target_words_romance.json", "w") as f:
    json.dump(all_categories_romance, f)


all_categories_sci_fi = {
    "high_competence": high_competence_sci_fi,
    "low_competence": low_competence_sci_fi,
    "high_warmth": high_warmth_sci_fi,
    "low_warmth": low_warmth_sci_fi
}

# save as json
with open(TARGET_WORDS_DIR / "target_words_sci_fi.json", "w") as f:
    json.dump(all_categories_sci_fi, f)

all_categories_literary_fiction = {
    "high_competence": high_competence_literary_fiction,
    "low_competence": low_competence_literary_fiction,
    "high_warmth": high_warmth_literary_fiction,
    "low_warmth": low_warmth_literary_fiction
}

# save as json
with open(TARGET_WORDS_DIR / "target_words_literary_fiction.json", "w") as f:
    json.dump(all_categories_literary_fiction, f)

### Calculate similarity

In [22]:
# Function to calculate the cosine similarity using gensim's similarity method
# this function does the same as cosine_sim, but at the model level instead of the vector level
def calculate_similarity(embeddings, word1, word2):
    if word1 in embeddings.wv and word2 in embeddings.wv:
        return embeddings.wv.similarity(word1, word2)
    else:
        return None  # if any word is missing

In [23]:
# save both versions with and without names in the gender vectors
gender_bias_results_romance_no_names = {}
gender_bias_results_sci_fi_no_names = {}
gender_bias_results_literary_fiction_no_names = {}

In [24]:
# Dictionary to hold results per seed
gender_bias_results_romance = {}
gender_bias_results_sci_fi = {}
gender_bias_results_literary_fiction = {}

##### Romance

In [25]:
# with names
for seed in seeds:
    model = updated_models["romance"][seed]  # use updated models with gender vectors

    data = []  # temporary list for this seed

    for dimension, target_words in all_categories_romance.items():
        for target_word in target_words:
            similarity_he = calculate_similarity(model, target_word, "he")
            similarity_she = calculate_similarity(model, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she

                data.append({
                    "dimension": dimension,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

    # Save results for this seed
    gender_bias_results_romance[seed] = data

In [26]:
# without names
for seed in seeds:
    model = updated_models_no_names["romance"][seed]  # use updated models with gender vectors

    data = []  # temporary list for this seed

    for dimension, target_words in all_categories_romance.items():
        for target_word in target_words:
            similarity_he = calculate_similarity(model, target_word, "he")
            similarity_she = calculate_similarity(model, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she

                data.append({
                    "dimension": dimension,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

    # Save results for this seed
    gender_bias_results_romance_no_names[seed] = data

##### Sci Fi

In [27]:
# with names
for seed in seeds:
    model = updated_models["sci_fi"][seed]  # use updated models with gender vectors

    data = []  # temporary list for this seed

    for dimension, target_words in all_categories_sci_fi.items():
        for target_word in target_words:
            similarity_he = calculate_similarity(model, target_word, "he")
            similarity_she = calculate_similarity(model, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she

                data.append({
                    "dimension": dimension,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

    # Save results for this seed
    gender_bias_results_sci_fi[seed] = data

In [28]:
# without names
for seed in seeds:
    model = updated_models_no_names["sci_fi"][seed]  # use updated models with gender vectors

    data = []  # temporary list for this seed

    for dimension, target_words in all_categories_sci_fi.items():
        for target_word in target_words:
            similarity_he = calculate_similarity(model, target_word, "he")
            similarity_she = calculate_similarity(model, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she

                data.append({
                    "dimension": dimension,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

    # Save results for this seed
    gender_bias_results_sci_fi_no_names[seed] = data

##### Literary Fiction

In [29]:
# with names
for seed in seeds:
    model = updated_models["literary_fiction"][seed]  # use your updated models with gender vectors

    data = []  # temporary list for this seed

    for dimension, target_words in all_categories_literary_fiction.items():
        for target_word in target_words:
            similarity_he = calculate_similarity(model, target_word, "he")
            similarity_she = calculate_similarity(model, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she

                data.append({
                    "dimension": dimension,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

    # Save results for this seed
    gender_bias_results_literary_fiction[seed] = data

In [30]:
# without names
for seed in seeds:
    model = updated_models_no_names["literary_fiction"][seed]  # use your updated models with gender vectors

    data = []  # temporary list for this seed

    for dimension, target_words in all_categories_literary_fiction.items():
        for target_word in target_words:
            similarity_he = calculate_similarity(model, target_word, "he")
            similarity_she = calculate_similarity(model, target_word, "she")

            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she

                data.append({
                    "dimension": dimension,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

    # Save results for this seed
    gender_bias_results_literary_fiction_no_names[seed] = data

### Save results

In [31]:
# Save results with names
for seed, data in gender_bias_results_romance.items():
    df = pd.DataFrame(data)
    df.to_csv(OUTPUT_DIR / f"romance_gender_bias_seed_{seed}.csv", index=False)
for seed, data in gender_bias_results_sci_fi.items():
    df = pd.DataFrame(data)
    df.to_csv(OUTPUT_DIR / f"sci_fi_gender_bias_seed_{seed}.csv", index=False)
for seed, data in gender_bias_results_literary_fiction.items():
    df = pd.DataFrame(data)
    df.to_csv(OUTPUT_DIR / f"literary_fiction_gender_bias_seed_{seed}.csv", index=False)

# Save results without names
for seed, data in gender_bias_results_romance_no_names.items():
    df = pd.DataFrame(data)
    df.to_csv(OUTPUT_DIR / f"romance_no_names_gender_bias_seed_{seed}.csv", index=False)
for seed, data in gender_bias_results_sci_fi_no_names.items():
    df = pd.DataFrame(data)
    df.to_csv(OUTPUT_DIR / f"sci_fi_no_names_gender_bias_seed_{seed}.csv", index=False)
for seed, data in gender_bias_results_literary_fiction_no_names.items():
    df = pd.DataFrame(data)
    df.to_csv(OUTPUT_DIR / f"literary_fiction_no_names_gender_bias_seed_{seed}.csv", index=False)


### Make averaged version

In [32]:
def average_gender_bias_across_seeds(gender_bias_dict):

    all_entries = []
    for seed, word_list in gender_bias_dict.items():
        for entry in word_list:
            entry_copy = entry.copy()
            entry_copy['seed'] = seed
            all_entries.append(entry_copy)
    
    df_long = pd.DataFrame(all_entries)
    
    df_avg = df_long.groupby(['dimension', 'word'], as_index=False).agg({
        'similarity_with_male_vector': 'mean',
        'similarity_with_female_vector': 'mean'
    })
    
    df_avg['gender_bias'] = df_avg['similarity_with_male_vector'] - df_avg['similarity_with_female_vector']
    
    return df_avg

In [33]:
df_avg_romance = average_gender_bias_across_seeds(gender_bias_results_romance)
df_avg_romance_no_names = average_gender_bias_across_seeds(gender_bias_results_romance_no_names)
df_avg_lit_fiction = average_gender_bias_across_seeds(gender_bias_results_literary_fiction)
df_avg_lit_fiction_no_names = average_gender_bias_across_seeds(gender_bias_results_literary_fiction_no_names)
df_avg_sci_fi = average_gender_bias_across_seeds(gender_bias_results_sci_fi)
df_avg_sci_fi_no_names = average_gender_bias_across_seeds(gender_bias_results_sci_fi_no_names)

In [34]:
df_avg_romance.to_csv(OUTPUT_DIR / "romance_gender_bias_averaged_seeds.csv", index=False)
df_avg_romance_no_names.to_csv(OUTPUT_DIR / "romance_no_names_gender_bias_averaged_seeds.csv", index=False)
df_avg_lit_fiction.to_csv(OUTPUT_DIR / "literary_fiction_gender_bias_averaged_seeds.csv", index=False)
df_avg_lit_fiction_no_names.to_csv(OUTPUT_DIR / "literary_fiction_no_names_gender_bias_averaged_seeds.csv", index=False)
df_avg_sci_fi.to_csv(OUTPUT_DIR / "sci_fi_gender_bias_averaged_seeds.csv", index=False)
df_avg_sci_fi_no_names.to_csv(OUTPUT_DIR / "sci_fi_no_names_gender_bias_averaged_seeds.csv", index=False)